# Object Detection on Custom Images

## Task:

Your task is to build and use an object detection model on custom images.

## Instructions:

1. Build a object detection model and load the pre-trained checkpoint.
2. Define the data augmentation for inference on custom images.
3. Define the visualization functions to draw bounding boxes on input images.
4. Inference on custom images and save visualization results.
5. Check the visualization results

In [1]:
# Import necessary libraries
import cv2
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

## Step 1: Build and Load Object Detection Model

In [2]:
# Define the configuration for the model 'fcos18'
from config.fcos_config import fcos_config
cfg = fcos_config['fcos18']

In [3]:
# Build the fcos18 model with configuration
from models.detector import build_model
device = torch.device("cpu")
model = build_model(version='fcos18',
                    cfg=cfg,
                    device=device, 
                    topk=100,
                    num_classes=80, 
                    trainable=False)

Build FCOS18 ...
Backbone: RESNET18
--pretrained: False
FPN: basic_fpn
Head: Decoupled Head


In [9]:
# Load checkpoint for the model
checkpoint = torch.load('checkpoints/fcos_r18_1x_31.3.pth', map_location='cpu', weights_only=False)
# Load checkpoint state dict
checkpoint_state_dict = checkpoint.pop("model")
# Load model state dict
# Add your implementation for load model state dict here
# TODO

model = model.to(device).eval()
model.eval()
print('Finished loading model!')

Finished loading model!


## Step 2: Define Inference Data Augmentation

In [5]:
# Define data augmentation for inference on custom images
from dataset.transforms import ValTransforms
transform = ValTransforms(min_size=cfg['test_min_size'], 
                          max_size=cfg['test_max_size'],
                          pixel_mean=cfg['pixel_mean'],
                          pixel_std=cfg['pixel_std'],
                          format=cfg['format'],
                          padding=cfg['val_padding'])

## Step 3: Define Visualization Functions

In [6]:
# Draw bounding box a single object
def plot_single_bbox_labels(img, bbox, label, cls_color, test_scale=0.4):
    # Plot bbox based on the predicted bbox
    # Add your implementation for getting (x1, y1), (x1, y2) from bbox
    # TODO

    x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
    cv2.rectangle(img, (x1, y1), (x2, y2), cls_color, 2)
    
    # Plot title bbox and put the text on the title bbox
    t_size = cv2.getTextSize(label, 0, fontScale=1, thickness=2)[0]
    cv2.rectangle(img, (x1, y1-t_size[1]), (int(x1 + t_size[0] * test_scale), y1), cls_color, -1)
    cv2.putText(img, label, (int(x1), int(y1 - 5)), 0, test_scale, (0, 0, 0), 1, lineType=cv2.LINE_AA)
    return img

In [7]:
# Draw bounding boxes for all objects in the image
from dataset.coco import coco_class_index, coco_class_labels
def plot_all_bbox_labels(img, bboxes, scores, cls_inds, class_colors, vis_thresh=0.3):
    ts = 0.4
    for i, bbox in enumerate(bboxes):
        if scores[i] > vis_thresh:
            # Define colors for bbox
            # Add your implementation for getting class colors and index based on cls_inds
            cls_color = class_colors[int(TODO)]
            cls_id = coco_class_index[int(TODO)]
            # Define text information for bbox
            # Add your implementation for adding score information into message
            mess = '%s: %.2f' % (coco_class_labels[cls_id], TODO)
            # Use the single object visualization function
            img = plot_single_bbox_labels(img, bbox, mess, cls_color, test_scale=ts)
    return img

It seems that the COCOAPI is not installed.


## Step 4: Inference on Custom Images and Save Visualizatons

In [8]:
# Define class color for bounding boxes
np.random.seed(0)
class_colors = [(np.random.randint(255),
                 np.random.randint(255),
                 np.random.randint(255)) for _ in range(80)]

# Define path to input custom images
path_to_img = 'dataset/demo/images'
print(f"Input images are from {path_to_img}.")

# Define path to save visualization results
save_path = os.path.join('det_results/images/image')
os.makedirs(save_path, exist_ok=True)
print(f"Results will be saved in {save_path}.")

# Main inference function
for i, img_id in enumerate(os.listdir(path_to_img)):
    # Load input images
    image = cv2.imread(path_to_img + '/' + img_id, cv2.IMREAD_COLOR)
    orig_h, orig_w, _ = image.shape
    orig_size = np.array([[orig_w, orig_h, orig_w, orig_h]])

    # Conduct inference data augmentation
    x = transform(image)[0]
    x = x.unsqueeze(0).to(device)
    
    # Inference on Images
    with torch.no_grad():
        # Add your implementation for obtaining bboxes, scores, cls_inds of the input image
        bboxes, scores, cls_inds = TODO

    # Rescale the predictions
    if transform.padding:
        # The input image is padded with 0 on the short side, aligning with the long side.
        bboxes *= max(orig_h, orig_w)
    else:
        # The input image is not padded.
        bboxes *= orig_size
        
    # Clip the bbox to ensure their ranges in the image
    bboxes[..., [0, 2]] = np.clip(bboxes[..., [0, 2]], a_min=0., a_max=orig_w)
    bboxes[..., [1, 3]] = np.clip(bboxes[..., [1, 3]], a_min=0., a_max=orig_h)

    # Visualizae the bbox predictions with the input image
    # Add your implementation for visualizing bboxes with the defined visualization function plot_all_bbox_labels()
    img_processed = plot_all_bbox_labels(TODO,
                                        vis_thresh=cfg['test_score_thresh'])
    
    # Save visualization results
    cv2.imwrite(os.path.join(save_path, str(i).zfill(6)+'.jpg'), img_processed)
    cv2.waitKey(0)
    print(f"Visualization results of {img_id} are saved.")


Input images are from dataset/demo/images.
Results will be saved in det_results/images/image.


/home/gary/miniconda3/envs/gs/lib/python3.8/site-packages/torch/functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /opt/conda/conda-bld/pytorch_1724789115405/work/aten/src/ATen/native/TensorShape.cpp:3609.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Visualization results of 000000000872.jpg are saved.
Visualization results of 000000001490.jpg are saved.
Visualization results of 000000002157.jpg are saved.
Visualization results of 000000001761.jpg are saved.
Visualization results of 000000001532.jpg are saved.
Visualization results of 000000000776.jpg are saved.
Visualization results of 000000000802.jpg are saved.
Visualization results of 000000001675.jpg are saved.
Visualization results of 000000001000.jpg are saved.
Visualization results of 000000001353.jpg are saved.
Visualization results of 000000001503.jpg are saved.
Visualization results of 000000002149.jpg are saved.
Visualization results of 000000000724.jpg are saved.
Visualization results of 000000001296.jpg are saved.
Visualization results of 000000001993.jpg are saved.
Visualization results of 000000001268.jpg are saved.
Visualization results of 000000000885.jpg are saved.
Visualization results of 000000000785.jpg are saved.
Visualization results of 000000001818.jpg are 

## Step 5: Check the visualization results

1. Check the visualization results in the path 'det_results/images/image'